In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes boto3

import os, json, boto3, torch
from google.colab import userdata
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
BUCKET = "legai-ai-embeddings"
SAMPLE_KEY = "finetune/sampled_contexts_5000.jsonl"
PART_PREFIX = "finetune/qa_parts"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

LOCAL_SAMPLE = "/content/sampled_contexts_5000.jsonl"
PART_SIZE = 100
BATCH_SIZE = 4

s3 = boto3.client("s3", aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID"), aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY"), region_name="us-east-1")

In [ ]:
s3.download_file(BUCKET, SAMPLE_KEY, LOCAL_SAMPLE)

with open(LOCAL_SAMPLE, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("Samples:", len(records))

Samples: 5000


In [ ]:
# if not torch.cuda.is_available():
#     raise RuntimeError("Enable GPU runtime in Colab first.")

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()

print("Model loaded")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [ ]:
SYSTEM_PROMPT = """
You are creating supervised fine-tuning data for LegaiAI,
a retrieval-augmented legal information system.

You will receive a legal text passage.

Create exactly ONE useful question that can be answered
directly from the supplied passage.

Then write a clear answer using ONLY information supported
by the passage.

Requirements:

- Do not use outside legal knowledge.
- Do not invent statutes, cases, citations, dates, parties,
  requirements, or facts.
- The answer must be grounded in the supplied context.
- The question should be natural and useful to a legal RAG user.
- Avoid trivial questions such as asking what the document says.
- Do not mention that you are generating training data.
- If the passage cannot support a useful question, return:
  {"skip": true}

Otherwise return ONLY valid JSON in exactly this structure:

{
  "question": "...",
  "answer": "..."
}
"""

In [ ]:
def make_prompt(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
     {"role": "user", "content": f"LEGAL CONTEXT:\n{text}\n\nCreate one grounded question and answer."}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def parse_answer(text):
    text = text.replace("```json", "").replace("```", "").strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except:
        return None
    if data.get("skip"):
        return None
    if not data.get("question") or not data.get("answer"):
        return None
    return data

In [ ]:
def generate_batch(batch):
    prompts = [make_prompt(x["text"]) for x in batch]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1800, add_special_tokens=False).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    generated = outputs[:, inputs["input_ids"].shape[1]:]
    answers = tokenizer.batch_decode(generated, skip_special_tokens=True)

    results = []

    for record, output in zip(batch, answers):
        qa = parse_answer(output)

        if qa:
            results.append({
                "vector_id": record.get("vector_id"),
                "source": record.get("source"),
                "context": record["text"],
                "question": qa["question"],
                "answer": qa["answer"]
            })

    return results

In [ ]:
test = generate_batch(records[:1])

for x in test:
    print(json.dumps(x, indent=2, ensure_ascii=False))
    print("=" * 80)

{
  "vector_id": 740882,
  "source": "csv2",
  "context": "Rotor Drive System (1) At least three 10-hour cycles required by this section must be conducted with transmission and gearbox lubricant temperatures, at the location prescribed for measurement, not lower than the maximum operating temperature for which approval is requested;",
  "question": "What is the minimum requirement for the number of 10-hour cycles that must be conducted?",
  "answer": "At least three 10-hour cycles"
}


In [ ]:
def exists(key):
    try:
        s3.head_object(Bucket=BUCKET, Key=key)
        return True
    except:
        return False

In [ ]:
def process_part(part_id, rows):
    key = f"{PART_PREFIX}/part_{part_id:04d}.jsonl"

    if exists(key):
        print(f"Part {part_id} already done")
        return

    results = []

    for i in range(0, len(rows), BATCH_SIZE):
        results.extend(generate_batch(rows[i:i + BATCH_SIZE]))

        if i % 20 == 0:
          os.system("nvidia-smi")

    local = f"/content/part_{part_id:04d}.jsonl"

    with open(local, "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    s3.upload_file(local, BUCKET, key)
    os.remove(local)

    print(f"Part {part_id} done | Q&A: {len(results)}")

In [ ]:
import math
total_parts = math.ceil(len(records) / PART_SIZE)

for part_id in range(total_parts):
    start = part_id * PART_SIZE
    end = min(start + PART_SIZE, len(records))
    process_part(part_id, records[start:end])

Part 0 already done
Part 1 already done
Part 2 already done
Part 3 done | Q&A: 94
Part 4 done | Q&A: 100
Part 5 done | Q&A: 99
Part 6 done | Q&A: 97
Part 7 done | Q&A: 99
Part 8 done | Q&A: 86
Part 9 done | Q&A: 98
Part 10 done | Q&A: 96
Part 11 done | Q&A: 100
Part 12 done | Q&A: 100
Part 13 done | Q&A: 97
Part 14 done | Q&A: 96
Part 15 done | Q&A: 100
Part 16 done | Q&A: 100
Part 17 done | Q&A: 97
Part 18 done | Q&A: 87
Part 19 done | Q&A: 100
Part 20 done | Q&A: 86
Part 21 done | Q&A: 91
Part 22 done | Q&A: 94
Part 23 done | Q&A: 99
Part 24 done | Q&A: 99
Part 25 done | Q&A: 99
Part 26 done | Q&A: 99
Part 27 done | Q&A: 98
Part 28 done | Q&A: 100
Part 29 done | Q&A: 93
Part 30 done | Q&A: 99
Part 31 done | Q&A: 100
Part 32 done | Q&A: 98
Part 33 done | Q&A: 100
Part 34 done | Q&A: 97
Part 35 done | Q&A: 98
Part 36 done | Q&A: 94
Part 37 done | Q&A: 100
Part 38 done | Q&A: 99
Part 39 done | Q&A: 90
Part 40 done | Q&A: 93
Part 41 done | Q&A: 100
Part 42 done | Q&A: 100
Part 43 done | 

In [ ]:
ALL_FILE = "/content/legal_rag_all.jsonl"

with open(ALL_FILE, "w", encoding="utf-8") as out:
    for part_id in range(total_parts):
        key = f"{PART_PREFIX}/part_{part_id:04d}.jsonl"

        if not exists(key):
            continue

        obj = s3.get_object(Bucket=BUCKET, Key=key)
        text = obj["Body"].read().decode("utf-8")

        out.write(text)

s3.upload_file(ALL_FILE, BUCKET, "finetune/legal_rag_all.jsonl")

print("Q&A dataset created")

Q&A dataset created


In [ ]:
import random

with open(ALL_FILE, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

random.seed(42)
random.shuffle(data)

split = int(len(data) * 0.95)
train_data = data[:split]
val_data = data[split:]

TRAIN_FILE = "/content/legal_rag_train.jsonl"
VAL_FILE = "/content/legal_rag_validation.jsonl"

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for row in train_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(VAL_FILE, "w", encoding="utf-8") as f:
    for row in val_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

s3.upload_file(TRAIN_FILE, BUCKET, "finetune/legal_rag_train.jsonl")
s3.upload_file(VAL_FILE, BUCKET, "finetune/legal_rag_validation.jsonl")

print("Train:", len(train_data))
print("Validation:", len(val_data))

Train: 4611
Validation: 243
